# Step 4 – Local (one-at-a-time) sensitivity analysis

For each farm the nominal point is the optimum for `LOCAL_METRIC` (step 1) and the farm's own herd. Each parameter is varied across its bound (`LOCAL_N_POINTS` values; T on a log scale, integers at every value) and herd size across `HS_GRID`, with the others held at nominal. Every value is simulated `LOCAL_REPS` times for `SA_HORIZON_DAYS`. The same random seeds are used at every value, so differences from nominal are paired by replicate. Run step 1 first.

**Outputs** (`results/sensitivity/`):

| File | Content |
| --- | --- |
| `local_runs.csv` | One row per farm, parameter, value and replicate |
| `local_summary.csv` | Per farm, parameter, value and outcome: mean, 2.5/97.5 percentiles, and paired difference from nominal |

In [1]:
import time
import warnings

import numpy as np
import pandas as pd

import ab_sim as m
import common as c
import config as cfg

from workers import local_task

## Functions

In [2]:
def grid_values(p: str) -> np.ndarray:
    """Values tested for parameter p across its bound in config.BOUNDS."""
    lo, hi = cfg.BOUNDS[p]
    if p in cfg.INTEGER_PARAMS:
        return np.arange(int(lo), int(hi) + 1)
    if p in cfg.LOG_SCALE_PARAMS:
        return np.geomspace(lo, hi, cfg.LOCAL_N_POINTS)
    return np.linspace(lo, hi, cfg.LOCAL_N_POINTS)

In [3]:
def paired_summary(runs: pd.DataFrame, nominal_values: dict) -> pd.DataFrame:
    """Mean and percentiles of each outcome per value, and the replicate-paired difference
    from the nominal run."""
    nom = runs[runs.parameter == "nominal"].set_index(["farm", "rep"])
    var = runs[runs.parameter != "nominal"]
    rows = []
    for (farm, param, value), g in var.groupby(["farm", "parameter", "value"]):
        g = g.set_index("rep")
        base = nom.loc[farm].loc[g.index]
        for o in c.OUTCOMES:
            d = g[o] - base[o]
            rows.append({"farm": farm, "parameter": param, "value": value,
                         "nominal_value": nominal_values[farm][param], "outcome": o,
                         "mean": g[o].mean(), "p2_5": g[o].quantile(0.025),
                         "p97_5": g[o].quantile(0.975), "nominal_mean": base[o].mean(),
                         "diff_mean": d.mean(), "diff_p2_5": d.quantile(0.025),
                         "diff_p97_5": d.quantile(0.975)})
    return pd.DataFrame(rows)

## Nominal points

In [4]:
c.ensure_dirs()
OUT = c.DIRS["sensitivity"]
farms = m.load_inputs(cfg)
c.farm_burn_ins(farms, verbose=False)
c.grid_burn_ins(verbose=False)

opt = pd.read_csv(c.DIRS["optimization"] / "optimal_parameters.csv")
opt = opt[opt.metric == cfg.LOCAL_METRIC]
best = opt.loc[opt.groupby("farm")["loss"].idxmin()].set_index("farm")
nominal_values = {farm: {p: best.loc[farm, p] for p in c.PARAMS} | {"HS": f["target_lactating"]}
                  for farm, f in farms.items()}
pd.DataFrame(nominal_values).T

,T,R,I,IC,HS
Farm1,0.001571,0.066749,1.0,1.0,800.0
Farm2,0.002210,0.079428,1.0,1.0,500.0
Farm3,0.002640,0.052711,1.0,5.0,2500.0
Farm4,0.000577,0.079753,2.0,8.0,3000.0


## Simulate

In [5]:
tasks = []
for farm, f in farms.items():
    nom = {p: best.loc[farm, p] for p in c.PARAMS}
    for rep in range(cfg.LOCAL_REPS):
        tasks.append((farm, "nominal", np.nan, nom, farm, rep))
        for p in c.PARAMS:
            tasks += [(farm, p, float(v), nom, farm, rep) for v in grid_values(p)]
        tasks += [(farm, "HS", float(hs), nom, hs, rep) for hs in cfg.HS_GRID]
print(f"Local sensitivity: {len(tasks)} simulations of {cfg.SA_HORIZON_DAYS} days")

t0 = time.time()
runs = pd.DataFrame(c.pool_map(local_task, tasks, include_grid=True, chunksize=8))
print(f"done in {time.time() - t0:.0f} s")
runs["nominal_value"] = [nominal_values[r.farm].get(r.parameter, np.nan) for r in runs.itertuples()]
runs.to_csv(OUT / "local_runs.csv", index=False)

Local sensitivity: 4880 simulations of 100 days
done in 16 s


## Summary

In [6]:
local = paired_summary(runs, nominal_values)
local.to_csv(OUT / "local_summary.csv", index=False)
local.head(20)

,farm,parameter,value,nominal_value,outcome,mean,p2_5,p97_5,nominal_mean,diff_mean,diff_p2_5,diff_p97_5
0,Farm1,HS,500.0,800.0,peak_daily_cases,3.900000,0.000000,10.525000,15.750000,-11.850000,-28.625000,1.575000
1,Farm1,HS,500.0,800.0,cumulative_cases,70.450000,0.000000,195.825000,264.650000,-194.200000,-492.975000,29.925000
2,Farm1,HS,500.0,800.0,cumulative_exposed,75.750000,0.000000,211.200000,290.900000,-215.150000,-525.150000,29.925000
3,Farm1,HS,500.0,800.0,attack_rate,0.135305,0.001762,0.374040,0.316748,-0.181443,-0.569748,0.053874
4,Farm1,HS,500.0,800.0,final_recovered,67.000000,1.000000,170.050000,257.850000,-190.850000,-457.250000,24.675000
5,Farm1,HS,500.0,800.0,outbreak_duration_days,51.100000,1.000000,100.000000,76.950000,-25.850000,-99.000000,43.575000
6,Farm1,HS,1000.0,800.0,peak_daily_cases,30.550000,0.000000,47.525000,15.750000,14.800000,-16.275000,44.575000
7,Farm1,HS,1000.0,800.0,cumulative_cases,440.100000,0.000000,650.100000,264.650000,175.450000,-263.550000,581.600000
8,Farm1,HS,1000.0,800.0,cumulative_exposed,486.500000,0.000000,717.350000,290.900000,195.600000,-277.200000,631.450000
9,Farm1,HS,1000.0,800.0,attack_rate,0.425941,0.000872,0.627928,0.316748,0.109194,-0.301192,0.551358
